In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 17:46:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/18 17:46:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/18 17:46:53 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/18 17:46:53 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
# Reading in the external datasets 
from urllib.request import urlretrieve
import os

# these are the ABS allocation files that will help us join postcode to SA2 via mesh block
# source: https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/access-and-downloads/allocation-files
ABS_FILES = {
    "MB_2021_AUST.xlsx": "https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/access-and-downloads/allocation-files/MB_2021_AUST.xlsx",
    "POA_2021_AUST.xlsx": "https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/access-and-downloads/allocation-files/POA_2021_AUST.xlsx",
}

output_relative_dir = '../../data/'

# data output directory is
abs_output_dir = output_relative_dir + 'raw_abs'
if not os.path.exists(abs_output_dir):
    os.makedirs(abs_output_dir)

for filename, url in ABS_FILES.items():
    output_path = f"{abs_output_dir}/{filename}"

    # Checks if the files have already been downloaded, skip if we've already downloaded it, 
    # every time the notebook is re-run for reproducibility checks
    if os.path.exists(output_path):
        print(f"Already downloaded: {filename}")
        continue

    print(f"Downloading {filename}...")
    urlretrieve(url, output_path)
    print(f"Completed {filename}")

Already downloaded: MB_2021_AUST.xlsx
Already downloaded: POA_2021_AUST.xlsx


In [3]:
import pandas as pd
excel_path_mb = f"{abs_output_dir}/MB_2021_AUST.xlsx"
pd.ExcelFile(excel_path_mb).sheet_names 

excel_path_poa = f"{abs_output_dir}/POA_2021_AUST.xlsx"
pd.ExcelFile(excel_path_poa).sheet_names 

['POA_2021_AUST']

In [4]:
mb_df = pd.read_excel(excel_path_mb, sheet_name='MB_2021_AUST')
mb_df.head()

,MB_CODE_2021,MB_CATEGORY_2021,CHANGE_FLAG_2021,CHANGE_LABEL_2021,SA1_CODE_2021,SA2_CODE_2021,SA2_NAME_2021,SA3_CODE_2021,SA3_NAME_2021,SA4_CODE_2021,SA4_NAME_2021,GCCSA_CODE_2021,GCCSA_NAME_2021,STATE_CODE_2021,STATE_NAME_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021
0,10000010000,Residential,0,No change,10901117207,109011172,Albury - East,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0209,http://linked.data.gov.au/dataset/asgsed3/MB/1...
1,10000021000,Commercial,0,No change,10901117612,109011176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0829,http://linked.data.gov.au/dataset/asgsed3/MB/1...
2,10000022000,Commercial,0,No change,10901117621,109011176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0388,http://linked.data.gov.au/dataset/asgsed3/MB/1...
3,10000023000,Commercial,0,No change,10901117621,109011176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0254,http://linked.data.gov.au/dataset/asgsed3/MB/1...
4,10000024000,Residential,0,No change,10901117613,109011176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0376,http://linked.data.gov.au/dataset/asgsed3/MB/1...


In [5]:

poa_df = pd.read_excel(excel_path_poa, sheet_name='POA_2021_AUST')
poa_df.head()

,MB_CODE_2021,POA_CODE_2021,POA_NAME_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021
0,70034860000,0800,0800,AUS,Australia,0.0434,http://linked.data.gov.au/dataset/asgsed3/MB/7...
1,73861000000,0800,0800,AUS,Australia,0.0218,http://linked.data.gov.au/dataset/asgsed3/MB/7...
2,70016404900,0800,0800,AUS,Australia,0.0877,http://linked.data.gov.au/dataset/asgsed3/MB/7...
3,70016404800,0800,0800,AUS,Australia,0.0419,http://linked.data.gov.au/dataset/asgsed3/MB/7...
4,70016404700,0800,0800,AUS,Australia,0.0931,http://linked.data.gov.au/dataset/asgsed3/MB/7...


In [6]:
# Explore dataset, missing values, data types
print("MB_2021 dataset shape", mb_df.shape)
print(mb_df.dtypes)

print("")
print(mb_df.isna().sum())
mb_df.head()

MB_2021 dataset shape (368286, 19)
MB_CODE_2021           object
MB_CATEGORY_2021       object
CHANGE_FLAG_2021        int64
CHANGE_LABEL_2021      object
SA1_CODE_2021          object
SA2_CODE_2021          object
SA2_NAME_2021          object
SA3_CODE_2021          object
SA3_NAME_2021          object
SA4_CODE_2021          object
SA4_NAME_2021          object
GCCSA_CODE_2021        object
GCCSA_NAME_2021        object
STATE_CODE_2021        object
STATE_NAME_2021        object
AUS_CODE_2021          object
AUS_NAME_2021          object
AREA_ALBERS_SQKM      float64
ASGS_LOCI_URI_2021     object
dtype: object

MB_CODE_2021            0
MB_CATEGORY_2021        0
CHANGE_FLAG_2021        0
CHANGE_LABEL_2021       0
SA1_CODE_2021           0
SA2_CODE_2021           0
SA2_NAME_2021           0
SA3_CODE_2021           0
SA3_NAME_2021           0
SA4_CODE_2021           0
SA4_NAME_2021           0
GCCSA_CODE_2021         0
GCCSA_NAME_2021         0
STATE_CODE_2021         0
STATE_NAME_2021 

,MB_CODE_2021,MB_CATEGORY_2021,CHANGE_FLAG_2021,CHANGE_LABEL_2021,SA1_CODE_2021,SA2_CODE_2021,SA2_NAME_2021,SA3_CODE_2021,SA3_NAME_2021,SA4_CODE_2021,SA4_NAME_2021,GCCSA_CODE_2021,GCCSA_NAME_2021,STATE_CODE_2021,STATE_NAME_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021
0,10000010000,Residential,0,No change,10901117207,109011172,Albury - East,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0209,http://linked.data.gov.au/dataset/asgsed3/MB/1...
1,10000021000,Commercial,0,No change,10901117612,109011176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0829,http://linked.data.gov.au/dataset/asgsed3/MB/1...
2,10000022000,Commercial,0,No change,10901117621,109011176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0388,http://linked.data.gov.au/dataset/asgsed3/MB/1...
3,10000023000,Commercial,0,No change,10901117621,109011176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0254,http://linked.data.gov.au/dataset/asgsed3/MB/1...
4,10000024000,Residential,0,No change,10901117613,109011176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,0.0376,http://linked.data.gov.au/dataset/asgsed3/MB/1...


In [7]:
# Explore dataset, missing values, data types
print("POA_2021 dataset shape", poa_df.shape)
print(poa_df.dtypes)

print("")
print(poa_df.isna().sum())
poa_df.head()

POA_2021 dataset shape (368286, 7)
MB_CODE_2021           object
POA_CODE_2021          object
POA_NAME_2021          object
AUS_CODE_2021          object
AUS_NAME_2021          object
AREA_ALBERS_SQKM      float64
ASGS_LOCI_URI_2021     object
dtype: object

MB_CODE_2021            0
POA_CODE_2021           0
POA_NAME_2021           0
AUS_CODE_2021           0
AUS_NAME_2021           0
AREA_ALBERS_SQKM      114
ASGS_LOCI_URI_2021      0
dtype: int64


,MB_CODE_2021,POA_CODE_2021,POA_NAME_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021
0,70034860000,0800,0800,AUS,Australia,0.0434,http://linked.data.gov.au/dataset/asgsed3/MB/7...
1,73861000000,0800,0800,AUS,Australia,0.0218,http://linked.data.gov.au/dataset/asgsed3/MB/7...
2,70016404900,0800,0800,AUS,Australia,0.0877,http://linked.data.gov.au/dataset/asgsed3/MB/7...
3,70016404800,0800,0800,AUS,Australia,0.0419,http://linked.data.gov.au/dataset/asgsed3/MB/7...
4,70016404700,0800,0800,AUS,Australia,0.0931,http://linked.data.gov.au/dataset/asgsed3/MB/7...


In [8]:
# Merging the two datasets on "MB_CODE_2021"
mb_poa_merged = mb_df.merge(
    poa_df,
    on='MB_CODE_2021',
    how='outer',
    indicator=True
)

print(mb_poa_merged['_merge'].value_counts())
print("Merged dataset shape: ", mb_poa_merged.shape)
print(mb_poa_merged.dtypes)
mb_poa_merged.head()


_merge
both          368286
left_only          0
right_only         0
Name: count, dtype: int64
Merged dataset shape:  (368286, 26)
MB_CODE_2021              object
MB_CATEGORY_2021          object
CHANGE_FLAG_2021           int64
CHANGE_LABEL_2021         object
SA1_CODE_2021             object
SA2_CODE_2021             object
SA2_NAME_2021             object
SA3_CODE_2021             object
SA3_NAME_2021             object
SA4_CODE_2021             object
SA4_NAME_2021             object
GCCSA_CODE_2021           object
GCCSA_NAME_2021           object
STATE_CODE_2021           object
STATE_NAME_2021           object
AUS_CODE_2021_x           object
AUS_NAME_2021_x           object
AREA_ALBERS_SQKM_x       float64
ASGS_LOCI_URI_2021_x      object
POA_CODE_2021             object
POA_NAME_2021             object
AUS_CODE_2021_y           object
AUS_NAME_2021_y           object
AREA_ALBERS_SQKM_y       float64
ASGS_LOCI_URI_2021_y      object
_merge                  category
dtype: obj

,MB_CODE_2021,MB_CATEGORY_2021,CHANGE_FLAG_2021,CHANGE_LABEL_2021,SA1_CODE_2021,SA2_CODE_2021,SA2_NAME_2021,SA3_CODE_2021,SA3_NAME_2021,SA4_CODE_2021,...,AUS_NAME_2021_x,AREA_ALBERS_SQKM_x,ASGS_LOCI_URI_2021_x,POA_CODE_2021,POA_NAME_2021,AUS_CODE_2021_y,AUS_NAME_2021_y,AREA_ALBERS_SQKM_y,ASGS_LOCI_URI_2021_y,_merge
0,10000009499,NOUSUALRESIDENCE,0,No change,19999949999,199999499,No usual address (NSW),19999,No usual address (NSW),199,...,Australia,NaN,http://linked.data.gov.au/dataset/asgsed3/MB/1...,9494,No usual address (Aust.),AUS,Australia,NaN,http://linked.data.gov.au/dataset/asgsed3/MB/1...,both
1,10000010000,Residential,0,No change,10901117207,109011172,Albury - East,10901,Albury,109,...,Australia,0.0209,http://linked.data.gov.au/dataset/asgsed3/MB/1...,2640,2640,AUS,Australia,0.0209,http://linked.data.gov.au/dataset/asgsed3/MB/1...,both
2,10000021000,Commercial,0,No change,10901117612,109011176,Lavington,10901,Albury,109,...,Australia,0.0829,http://linked.data.gov.au/dataset/asgsed3/MB/1...,2641,2641,AUS,Australia,0.0829,http://linked.data.gov.au/dataset/asgsed3/MB/1...,both
3,10000022000,Commercial,0,No change,10901117621,109011176,Lavington,10901,Albury,109,...,Australia,0.0388,http://linked.data.gov.au/dataset/asgsed3/MB/1...,2641,2641,AUS,Australia,0.0388,http://linked.data.gov.au/dataset/asgsed3/MB/1...,both
4,10000023000,Commercial,0,No change,10901117621,109011176,Lavington,10901,Albury,109,...,Australia,0.0254,http://linked.data.gov.au/dataset/asgsed3/MB/1...,2641,2641,AUS,Australia,0.0254,http://linked.data.gov.au/dataset/asgsed3/MB/1...,both


In [9]:
mb_poa_merged = mb_poa_merged.drop(columns=[
    'MB_CATEGORY_2021', 'CHANGE_FLAG_2021', 'CHANGE_LABEL_2021',
    'SA1_CODE_2021',
    'SA3_CODE_2021', 'SA3_NAME_2021',
    'SA4_CODE_2021', 'SA4_NAME_2021',
    'GCCSA_CODE_2021', 'GCCSA_NAME_2021',
    'STATE_CODE_2021', 'STATE_NAME_2021',
    'AUS_CODE_2021_x', 'AUS_NAME_2021_x', 'AREA_ALBERS_SQKM_x', 'ASGS_LOCI_URI_2021_x',
    'AUS_CODE_2021_y', 'AUS_NAME_2021_y', 'AREA_ALBERS_SQKM_y', 'ASGS_LOCI_URI_2021_y',
    '_merge',
])

mb_poa_merged.head()

,MB_CODE_2021,SA2_CODE_2021,SA2_NAME_2021,POA_CODE_2021,POA_NAME_2021
0,10000009499,199999499,No usual address (NSW),9494,No usual address (Aust.)
1,10000010000,109011172,Albury - East,2640,2640
2,10000021000,109011176,Lavington,2641,2641
3,10000022000,109011176,Lavington,2641,2641
4,10000023000,109011176,Lavington,2641,2641


In [10]:
# Making sure that the MB_CODE_2021, POA_CODE_2021 and SA2_CODE_2021 are strictly stored as string data types 
# instead of integers. This prevents future issues in reading the files 
mb_poa_merged['MB_CODE_2021'] = mb_poa_merged['MB_CODE_2021'].astype(str)
mb_poa_merged['SA2_CODE_2021'] = mb_poa_merged['SA2_CODE_2021'].astype(str)
mb_poa_merged['POA_CODE_2021'] = mb_poa_merged['POA_CODE_2021'].astype(str)

# save the cleaned MB and POA merged dataset to be reused in the next step 
mb_poa_merged.to_parquet(f"{abs_output_dir}/mb_poa_merged.parquet", index=False)

26/09/18 22:02:45 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 922236 ms exceeds timeout 120000 ms
26/09/18 22:02:45 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/18 22:02:48 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:707